In [1]:
import pandas as pd
import gc
import numpy as np

In [2]:
df3 = pd.read_pickle('../data/df3.pkl')

In [3]:
df3.info()
df3.shape # (7483231, 30)
#df3.head(20)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7483231 entries, 0 to 7483230
Data columns (total 30 columns):
 #   Column              Dtype         
---  ------              -----         
 0   BUS_SVC_NUM         float64       
 1   CRD_NUM             object        
 2   DEST_LOC_ID_NUM     float64       
 3   ENTRY_DT            datetime64[s] 
 4   ENTRY_TM            datetime64[ns]
 5   EXIT_DT             datetime64[s] 
 6   EXIT_TM             datetime64[ns]
 7   JRNY_ID_NUM         object        
 8   ORIG_LOC_ID_NUM     float64       
 9   RIDE_DISC_AMT       float64       
 10  RIDE_DIST_KM_CNT    float64       
 11  RIDE_FARE_AMT       float64       
 12  RIDE_ID_NUM         object        
 13  RIDE_MIN_CNT        float64       
 14  PATRON_CATG_ID_NUM  int64         
 15  TRNSPT_MODE_CD      int64         
 16  DEST_STATION_NAME   object        
 17  DEST_MRK_ID_NUM     float64       
 18  DEST_LATITUDE       float64       
 19  DEST_LONGITUDE      float64       
 20  DE

(7483231, 30)

### Information on the Patron Categories: 

- Student (Primary 1 to JC/Poly) --> 7 - 19 years old
- Adult --> 20 - 59 years old
- Senior Citizen --> 60 years and above

Average Walking Speeds by Age:
- Children (6-12 years): Around 3.0 to 4.5 km/h (1.9 to 2.8 mph)
- Adolescents (13-19 years): Around 4.0 to 5.0 km/h (2.5 to 3.1 mph)
- Adults (20-59 years): Around 5.0 to 6.0 km/h (3.1 to 3.7 mph)
- Older Adults (60+ years): Around 4.0 to 5.0 km/h (2.5 to 3.1 mph)

To calculate the walking speed of each group, I take the midpoint of the range. 
For Children and Adolescents, I took the midpoint of both ranges and took the average.

In [4]:
# We first filter for 3 relevant PATRON_CATG_ID_NUM.
patron_map = {1: 'Adult', 3: 'Student', 4: 'Senior Citizen'}
df3['PATRON_CATG_DESC_TXT'] = df3['PATRON_CATG_ID_NUM'].map(patron_map)

walking_speed_ms = {
    'Student': 4.125 / 3.6,      # (3.75 + 4.5) / 2 = 4.125 km/h → ~1.146 m/s
    'Adult': 5.5 / 3.6,          # ~1.528 m/s
    'Senior Citizen': 4.5 / 3.6  # 1.25 m/s
}

df3['walking_speed_ms'] = df3['PATRON_CATG_DESC_TXT'].map(walking_speed_ms)

# Rule Based Classification

#### 1. Binary Criteria
- Drop rows with missing critical data
- The journey stage has no subsequent recorded trips on the same day and is therefore treated as a journey termination. It is the last ride of the day for the commuter.
    - Implementation: df3 is already sorted by CRD_NUM and ENTRY_TM. Flag last row of each CRD_NUM
- 2 consecutive journey stages on the same bus service or train station indicates a return trip or intermediate activity. This should be classified as a new journey.
    - Implementation: Shift BUS_SVC_NUM and ORIG_LOC_ID_NUM down by 1 row to get the next stage's values, then flag rows where all 3 conditions are true simultaneously: same commuter, same bus service, and the alighting stop of stage 1 equals the boarding stop of stage 2.


In [5]:
df3 = df3.sort_values(["CRD_NUM", "ENTRY_TM"]).reset_index(drop=True)

In [6]:
df3["next_ENTRY_TM"] = df3.groupby("CRD_NUM")["ENTRY_TM"].shift(-1)
df3["next_ORIG_LOC_ID_NUM"] = df3.groupby("CRD_NUM")["ORIG_LOC_ID_NUM"].shift(-1)
df3["next_BUS_SVC_NUM"] = df3.groupby("CRD_NUM")["BUS_SVC_NUM"].shift(-1)
df3["next_TRNSPT_MODE_CD"] = df3.groupby("CRD_NUM")["TRNSPT_MODE_CD"].shift(-1)

In [7]:
df3.shape

(7483231, 36)

In [8]:
df3["is_last_stage"] = df3["next_ENTRY_TM"].isna()

df3['missing_info'] = (
    df3['ENTRY_TM'].isna() |
    df3['EXIT_TM'].isna() |
    df3['ORIG_LATITUDE'].isna() |
    df3['ORIG_LONGITUDE'].isna() |
    df3['DEST_LATITUDE'].isna() |
    df3['DEST_LONGITUDE'].isna() |
    df3["ORIG_LOC_ID_NUM"].isna() |
    df3["DEST_LOC_ID_NUM"].isna()
)

In [9]:
#create next bus and next station columns by shifting
# flags consecutive rides on the same bus service number/same station
df3["same_bus_service"] = (
    df3["BUS_SVC_NUM"].notna() &
    df3["next_BUS_SVC_NUM"].notna() &
    (df3["BUS_SVC_NUM"] == df3["next_BUS_SVC_NUM"])
)
df3["same_station_consecutive"] = (
    df3["DEST_STATION_NAME"].notna() &
    df3["next_orig_station"].notna() &
    (df3["DEST_STATION_NAME"] == df3["next_orig_station"])
)

In [10]:
df3["return_or_intermediate"] = (
    df3["same_bus_service"] |
    df3["same_station_consecutive"]
)

In [11]:
df3["journey_termination_flag"] = (
    df3["is_last_stage"] |
    df3["missing_info"] |
    df3["return_or_intermediate"]
)

In [12]:
df3['journey_termination_flag'].value_counts()


journey_termination_flag
True     4493025
False    2990206
Name: count, dtype: int64

In [13]:
# Summary counts
print("Unique commuters:", df3["CRD_NUM"].nunique())
print("Last stages:", df3["is_last_stage"].sum())

print("\nMissing info:")
print(df3["missing_info"].value_counts(dropna=False))

print("\nSame bus service:")
print(df3["same_bus_service"].value_counts(dropna=False))

print("\nSame station consecutive:")
print(df3["same_station_consecutive"].value_counts(dropna=False))

print("\nReturn / intermediate:")
print(df3["return_or_intermediate"].value_counts(dropna=False))

print("\nJourney termination flag:")
print(df3["journey_termination_flag"].value_counts(dropna=False))

Unique commuters: 2462129
Last stages: 2462129

Missing info:
missing_info
False    7414732
True       68499
Name: count, dtype: int64

Same bus service:
same_bus_service
False    6931342
True      551889
Name: count, dtype: int64

Same station consecutive:
same_station_consecutive
False    5952424
True     1530807
Name: count, dtype: int64

Return / intermediate:
return_or_intermediate
False    5486609
True     1996622
Name: count, dtype: int64

Journey termination flag:
journey_termination_flag
True     4493025
False    2990206
Name: count, dtype: int64


In [14]:
# If you want to see flag rows
'''
display(
    df3.loc[
        df3["journey_termination_flag"],
        [
            "CRD_NUM",
            "ENTRY_TM",
            "EXIT_TM",
            "ORIG_STATION_NAME",
            "DEST_STATION_NAME",
            "next_orig_station",
            "BUS_SVC_NUM",
            "next_BUS_SVC_NUM",
            "is_last_stage",
            "missing_info",
            "same_bus_service",
            "same_station_consecutive",
            "return_or_intermediate",
            "journey_termination_flag"
        ]
    ].head(20)
)
'''

'\ndisplay(\n    df3.loc[\n        df3["journey_termination_flag"],\n        [\n            "CRD_NUM",\n            "ENTRY_TM",\n            "EXIT_TM",\n            "ORIG_STATION_NAME",\n            "DEST_STATION_NAME",\n            "next_orig_station",\n            "BUS_SVC_NUM",\n            "next_BUS_SVC_NUM",\n            "is_last_stage",\n            "missing_info",\n            "same_bus_service",\n            "same_station_consecutive",\n            "return_or_intermediate",\n            "journey_termination_flag"\n        ]\n    ].head(20)\n)\n'

In [15]:
# If you want to see transfers
'''
display(
    df3.loc[
        ~df3["journey_termination_flag"],
        [
            "CRD_NUM",
            "ENTRY_TM",
            "EXIT_TM",
            "ORIG_STATION_NAME",
            "DEST_STATION_NAME",
            "next_orig_station",
            "BUS_SVC_NUM",
            "next_BUS_SVC_NUM",
            "is_last_stage",
            "missing_info",
            "same_bus_service",
            "same_station_consecutive",
            "return_or_intermediate",
            "journey_termination_flag"
        ]
    ].head(20)
)
'''

'\ndisplay(\n    df3.loc[\n        ~df3["journey_termination_flag"],\n        [\n            "CRD_NUM",\n            "ENTRY_TM",\n            "EXIT_TM",\n            "ORIG_STATION_NAME",\n            "DEST_STATION_NAME",\n            "next_orig_station",\n            "BUS_SVC_NUM",\n            "next_BUS_SVC_NUM",\n            "is_last_stage",\n            "missing_info",\n            "same_bus_service",\n            "same_station_consecutive",\n            "return_or_intermediate",\n            "journey_termination_flag"\n        ]\n    ].head(20)\n)\n'

In [16]:
# Reasons why flagged.
df3["journey_termination_reason"] = np.select(
    [
        df3["is_last_stage"],
        df3["missing_info"],
        df3["return_or_intermediate"]
    ],
    [
        "last_stage",
        "missing_info",
        "return_or_intermediate"
    ],
    default="candidate_transfer"
)

In [17]:
print(df3["journey_termination_reason"].value_counts(dropna=False))

journey_termination_reason
candidate_transfer        2990206
last_stage                2462129
return_or_intermediate    1985090
missing_info                45806
Name: count, dtype: int64


In [18]:
#df3.head(50)

In [19]:
df4 = df3.copy()

#define full journey sequence
df4['full_journey_seq'] = (
    df4.groupby('CRD_NUM')['journey_termination_flag']
       .shift(fill_value=False)
       .groupby(df4['CRD_NUM'])
       .cumsum()
)

#first row of each full journey
journey_first = (
    df4.drop_duplicates(subset=['CRD_NUM', 'full_journey_seq'], keep='first')
       [['CRD_NUM', 'full_journey_seq', 'ORIG_STATION_NAME', 'ENTRY_TM']]
       .rename(columns={
           'ORIG_STATION_NAME': 'orig_journey_station',
           'ENTRY_TM': 'entry_journey_tm'
       })
)

#last row of each full journey
journey_last = (
    df4.drop_duplicates(subset=['CRD_NUM', 'full_journey_seq'], keep='last')
       [['CRD_NUM', 'full_journey_seq',
         'DEST_STATION_NAME', 'EXIT_TM',
         'next_orig_station', 'walk_distance']]
       .rename(columns={
           'DEST_STATION_NAME': 'dest_journey_station',
           'EXIT_TM': 'exit_journey_tm',
           'next_orig_station': 'next_orig_journey_location',
           'walk_distance': 'walk_to_next_journey_distance'
       })
)

#combine first + last journey info
journey_info = journey_first.merge(
    journey_last,
    on=['CRD_NUM', 'full_journey_seq'],
    how='inner'
)

#merge back to every row
df4 = df4.merge(
    journey_info,
    on=['CRD_NUM', 'full_journey_seq'],
    how='left'
)


### Create "median" time gap for each bucket of walking distance (in 50m intervals). 
Group by patron cat


In [20]:
'''# Calculate time gap mins

# Shift next entry time within the same commuter group
df3['next_ENTRY_TM'] = df3.groupby('CRD_NUM')['ENTRY_TM'].shift(-1)

# Time gap = next stage's entry time minus current stage's exit time (in mins)
df3['time_gap_mins'] = (
    df3['next_ENTRY_TM'] - df3['EXIT_TM']
).dt.total_seconds() / 60'''


"# Calculate time gap mins\n\n# Shift next entry time within the same commuter group\ndf3['next_ENTRY_TM'] = df3.groupby('CRD_NUM')['ENTRY_TM'].shift(-1)\n\n# Time gap = next stage's entry time minus current stage's exit time (in mins)\ndf3['time_gap_mins'] = (\n    df3['next_ENTRY_TM'] - df3['EXIT_TM']\n).dt.total_seconds() / 60"

In [21]:
'''bins = list(range(0, 1050, 50))
labels = [f"{i}-{i+50}m" for i in range(0, 1000, 50)]

# Create as object dtype so we can freely assign '0m'
df3['walk_dist_bucket'] = pd.cut(
    df3['walk_distance'],
    bins=bins,
    labels=labels,
    right=False
).astype(object)

# Override: exact 0 gets its own bucket. 0 is removed from 0-50m bucket.
df3.loc[df3['walk_distance'] == 0, 'walk_dist_bucket'] = '0m'

def first_10pct_median(x):
    x_sorted = x.sort_values()
    n_keep = max(1, int(np.ceil(len(x_sorted) * 0.10)))  # keep at least 1 row
    x_top10pct = x_sorted.iloc[:n_keep]
    return x_top10pct.median()

bucket_medians = (
    df3.dropna(subset=['walk_distance', 'time_gap_mins', 'PATRON_CATG_DESC_TXT'])
    .groupby(['walk_dist_bucket', 'PATRON_CATG_DESC_TXT'], observed=True)['time_gap_mins']
    .agg(
        count='count',
        median_time_gap_mins='median',                  # original median
        median_first10pct_time_gap_mins=first_10pct_median
    )
    .reset_index()
)

print(bucket_medians.to_string())'''

'bins = list(range(0, 1050, 50))\nlabels = [f"{i}-{i+50}m" for i in range(0, 1000, 50)]\n\n# Create as object dtype so we can freely assign \'0m\'\ndf3[\'walk_dist_bucket\'] = pd.cut(\n    df3[\'walk_distance\'],\n    bins=bins,\n    labels=labels,\n    right=False\n).astype(object)\n\n# Override: exact 0 gets its own bucket. 0 is removed from 0-50m bucket.\ndf3.loc[df3[\'walk_distance\'] == 0, \'walk_dist_bucket\'] = \'0m\'\n\ndef first_10pct_median(x):\n    x_sorted = x.sort_values()\n    n_keep = max(1, int(np.ceil(len(x_sorted) * 0.10)))  # keep at least 1 row\n    x_top10pct = x_sorted.iloc[:n_keep]\n    return x_top10pct.median()\n\nbucket_medians = (\n    df3.dropna(subset=[\'walk_distance\', \'time_gap_mins\', \'PATRON_CATG_DESC_TXT\'])\n    .groupby([\'walk_dist_bucket\', \'PATRON_CATG_DESC_TXT\'], observed=True)[\'time_gap_mins\']\n    .agg(\n        count=\'count\',\n        median_time_gap_mins=\'median\',                  # original median\n        median_first10pct_time_g

In [22]:
'''import matplotlib.pyplot as plt
import numpy as np

# =========================================================
# 1. Convert distance bucket → numeric midpoint
# =========================================================
def bucket_midpoint(bucket):
    if bucket == '0m':
        return 0
    try:
        start, end = bucket.replace('m', '').split('-')
        return (float(start) + float(end)) / 2
    except:
        return np.nan

bucket_medians['distance_mid_m'] = bucket_medians['walk_dist_bucket'].apply(bucket_midpoint)


# =========================================================
# 2. Clean data for plotting
# =========================================================
plot_df = bucket_medians.dropna(subset=[
    'distance_mid_m',
    'median_first10pct_time_gap_mins'
]).copy()

# Optional: remove 0m (usually weird case)
plot_df = plot_df[plot_df['distance_mid_m'] > 0]


# =========================================================
# 3. Plot
# =========================================================
plt.figure()

for patron, g in plot_df.groupby('PATRON_CATG_DESC_TXT'):
    g = g.sort_values('distance_mid_m')
    
    plt.plot(
        g['distance_mid_m'],
        g['median_first10pct_time_gap_mins'],
        marker='o',
        label=patron
    )

plt.xlabel('Distance (m)')
plt.ylabel('Median Time Gap (mins)')
plt.title('Median Time Gap vs Walking Distance (First 10% Fastest)')
plt.legend()
plt.grid(True)

plt.show()'''

"import matplotlib.pyplot as plt\nimport numpy as np\n\n# =========================================================\n# 1. Convert distance bucket → numeric midpoint\n# =========================================================\ndef bucket_midpoint(bucket):\n    if bucket == '0m':\n        return 0\n    try:\n        start, end = bucket.replace('m', '').split('-')\n        return (float(start) + float(end)) / 2\n    except:\n        return np.nan\n\nbucket_medians['distance_mid_m'] = bucket_medians['walk_dist_bucket'].apply(bucket_midpoint)\n\n\n# =========================================================\n# 2. Clean data for plotting\n# =========================================================\nplot_df = bucket_medians.dropna(subset=[\n    'distance_mid_m',\n    'median_first10pct_time_gap_mins'\n]).copy()\n\n# Optional: remove 0m (usually weird case)\nplot_df = plot_df[plot_df['distance_mid_m'] > 0]\n\n\n# =========================================================\n# 3. Plot\n# =========

In [23]:
'''# Map median back to df3 by bucket + patron category
df3 = df3.merge(
    bucket_medians[['walk_dist_bucket', 'PATRON_CATG_DESC_TXT', 'median_time_gap_mins']],
    on=['walk_dist_bucket', 'PATRON_CATG_DESC_TXT'],
    how='left'
)

print(df3['median_time_gap_mins'].isna().sum(), "rows with no median (null bucket or patron category)")'''

'# Map median back to df3 by bucket + patron category\ndf3 = df3.merge(\n    bucket_medians[[\'walk_dist_bucket\', \'PATRON_CATG_DESC_TXT\', \'median_time_gap_mins\']],\n    on=[\'walk_dist_bucket\', \'PATRON_CATG_DESC_TXT\'],\n    how=\'left\'\n)\n\nprint(df3[\'median_time_gap_mins\'].isna().sum(), "rows with no median (null bucket or patron category)")'

In [24]:
'''# this not very useful right?
bucket_stats_combined = (
    df3.dropna(subset=['walk_distance', 'time_gap_mins'])
    .groupby('walk_dist_bucket', observed=True)['time_gap_mins']
    .agg(count='count', median_time_gap_mins='median')
    .reset_index()
)

print(bucket_stats_combined.to_string())'''

"# this not very useful right?\nbucket_stats_combined = (\n    df3.dropna(subset=['walk_distance', 'time_gap_mins'])\n    .groupby('walk_dist_bucket', observed=True)['time_gap_mins']\n    .agg(count='count', median_time_gap_mins='median')\n    .reset_index()\n)\n\nprint(bucket_stats_combined.to_string())"

#### 2. Temporal criteria: 
- The binary criteria gives us pairs of consecutive stages that could be transfers. 
- Temporal criteria finds the time gap between alighting from stage 1 and boarding stage 2
- Implementation: 
    - gap = ENTRY_TM of stage 2 - EXIT_TM of stage 1. 
    - Shift ENTRY_TM and TRNSPT_MODE_CD down by 1 row to get the next stage's values. Compute the time gap between current EXIT_TM and next ENTRY_TM. Assign threshold (15 or 45 mins) based on transfer type, then flag if gap exceeds it.
    - Flag if gap > allowance.
    - Allowance = Walking speed * walking distance



Describe.

In [25]:
# Calculate time gap mins

# Shift next entry time within the same commuter group
df3['next_ENTRY_TM'] = df3.groupby('CRD_NUM')['ENTRY_TM'].shift(-1)

# Time gap = next stage's entry time minus current stage's exit time (in mins)
df3['time_gap_mins'] = (
    df3['next_ENTRY_TM'] - df3['EXIT_TM']
).dt.total_seconds() / 60

In [26]:
# allowance (seconds) = walking distance (m) / walking speed (m/s), then convert to mins
# Rows with no walking distance → allowance is NaN  
df3['predicted_walking_time_mins'] = (
     df3['walk_distance'] / df3['walking_speed_ms']
 ) / 60

# True = gap exceeds allowance → new journey (not a transfer)
# If walk_distance is null → allowance is NaN → comparison returns NaN → fillna(True) marks as new journey
df3['waiting_time_plus_extra'] = df3['time_gap_mins'] - df3['predicted_walking_time_mins']

df3['waiting_time_allowed'] = 20
df3['extras_allowed'] = 0
df3['Total_allowance'] = df3['waiting_time_allowed'] + df3['extras_allowed']

df3['temporal_flag_reason'] = np.select(
    [
        df3['time_gap_mins'].isna(),
        df3['predicted_walking_time_mins'].isna(),
        df3['waiting_time_plus_extra'].isna(),
        df3['waiting_time_plus_extra'] > df3['Total_allowance'],
    ],
    [
        'null_time_gap',
        'null_predicted_walking_time',
        'null_waiting_time_plus_extra',
        'exceeds_total_allowance',
    ],
    default='within_total_allowance'
)

df3['exceeds_time_allowance'] = df3['temporal_flag_reason'] != 'within_total_allowance'

print(df3['exceeds_time_allowance'].value_counts())
print(f"\nTotal temporal new journey flags: {df3['exceeds_time_allowance'].sum():,}")
print(df3['temporal_flag_reason'].value_counts(dropna=False))

#print(f"Null walking distance: {df3['walk_distance'].isna().sum():,}")
#print(f"Total rows: {len(df3):,}")
#print(f"Null %: {df3['walk_distance'].isna().mean() * 100:.2f}%")'''

exceeds_time_allowance
True     5311208
False    2172023
Name: count, dtype: int64

Total temporal new journey flags: 5,311,208
temporal_flag_reason
null_time_gap                  2507790
exceeds_total_allowance        2492655
within_total_allowance         2172023
null_predicted_walking_time     310763
Name: count, dtype: int64


In [27]:
#define temporal-only journey sequence AFTER final temporal flag is ready
df3['temporal_full_journey_seq'] = (
    df3.groupby('CRD_NUM')['exceeds_time_allowance']
       .shift(fill_value=False)
       .groupby(df3['CRD_NUM'])
       .cumsum()
)

#first row of each temporal-only journey
temporal_journey_first = (
    df3.drop_duplicates(subset=['CRD_NUM', 'temporal_full_journey_seq'], keep='first')
       [['CRD_NUM', 'temporal_full_journey_seq', 'ORIG_STATION_NAME', 'ENTRY_TM']]
       .rename(columns={
           'ORIG_STATION_NAME': '(TEMPORAL_Only)_orig_journey',
           'ENTRY_TM': '(TEMPORAL_Only)_entry_journey_tm'
       })
)

#last row of each temporal-only journey
temporal_journey_last = (
    df3.drop_duplicates(subset=['CRD_NUM', 'temporal_full_journey_seq'], keep='last')
       [['CRD_NUM', 'temporal_full_journey_seq', 'DEST_STATION_NAME', 'EXIT_TM']]
       .rename(columns={
           'DEST_STATION_NAME': '(TEMPORAL_Only)_dest_journey',
           'EXIT_TM': '(TEMPORAL_Only)_exit_journey_tm'
       })
)

#combine first + last journey info
temporal_journey_info = temporal_journey_first.merge(
    temporal_journey_last,
    on=['CRD_NUM', 'temporal_full_journey_seq'],
    how='inner'
)

#merge back to df3
df3 = df3.merge(
    temporal_journey_info,
    on=['CRD_NUM', 'temporal_full_journey_seq'],
    how='left'
)

* Note this is how TRNSPRT_MODE_CD is mapped.
1    BUS
2    RTS
3    BUS-RTS (interchange)

In [28]:
'''# Flag: time_gap > median of its bucket+patron category → new journey
# Null median (no bucket or patron category match) → fillna(True) marks as new journey
df3['exceeds_time_allowance'] = (
    df3['time_gap_mins'] > df3['median_time_gap_mins']
).fillna(True)

print(df3['exceeds_time_allowance'].value_counts())
print(f"\nTotal temporal new journey flags: {df3['exceeds_time_allowance'].sum():,}")
print(f"Null median (no bucket/patron match): {df3['median_time_gap_mins'].isna().sum():,}")
print(f"Total rows: {len(df3):,}")'''

'# Flag: time_gap > median of its bucket+patron category → new journey\n# Null median (no bucket or patron category match) → fillna(True) marks as new journey\ndf3[\'exceeds_time_allowance\'] = (\n    df3[\'time_gap_mins\'] > df3[\'median_time_gap_mins\']\n).fillna(True)\n\nprint(df3[\'exceeds_time_allowance\'].value_counts())\nprint(f"\nTotal temporal new journey flags: {df3[\'exceeds_time_allowance\'].sum():,}")\nprint(f"Null median (no bucket/patron match): {df3[\'median_time_gap_mins\'].isna().sum():,}")\nprint(f"Total rows: {len(df3):,}")'

In [29]:
# # PREVIOUS WAY OF TEMPORAL

# # allowance (seconds) = walking distance (m) / walking speed (m/s), then convert to mins
# # Rows with no walking distance → allowance is NaN  
# df3['transfer_allowance_mins'] = (
#     df3['walk_distance'] / df3['walking_speed_ms']
# ) / 60

# # True = gap exceeds allowance → new journey (not a transfer)
# # If walk_distance is null → allowance is NaN → comparison returns NaN → fillna(True) marks as new journey
# df3['exceeds_time_allowance'] = (
#     (df3['time_gap_mins'] > df3['transfer_allowance_mins'])
# ).fillna(True)

# print(df3['exceeds_time_allowance'].value_counts())
# print(f"\nTotal temporal new journey flags: {df3['exceeds_time_allowance'].sum():,}")

# print(f"Null walking distance: {df3['walk_distance'].isna().sum():,}")
# print(f"Total rows: {len(df3):,}")
# print(f"Null %: {df3['walk_distance'].isna().mean() * 100:.2f}%")

## Using Binary and Temporal
- apply temporal criteria only after filtering transfers from binary

In [30]:
'''# Bucket medians excluding binary-flagged rows
bucket_medians_clean = (
    df3[df3['journey_termination_flag'] == False] # only rows that pass the binary criteria
    .dropna(subset=['walk_distance', 'time_gap_mins', 'PATRON_CATG_DESC_TXT'])
    .groupby(['walk_dist_bucket', 'PATRON_CATG_DESC_TXT'], observed=True)['time_gap_mins']
    .agg(count='count', median_time_gap_mins_clean='median')
    .reset_index()
)

print(bucket_medians_clean.to_string())'''

"# Bucket medians excluding binary-flagged rows\nbucket_medians_clean = (\n    df3[df3['journey_termination_flag'] == False] # only rows that pass the binary criteria\n    .dropna(subset=['walk_distance', 'time_gap_mins', 'PATRON_CATG_DESC_TXT'])\n    .groupby(['walk_dist_bucket', 'PATRON_CATG_DESC_TXT'], observed=True)['time_gap_mins']\n    .agg(count='count', median_time_gap_mins_clean='median')\n    .reset_index()\n)\n\nprint(bucket_medians_clean.to_string())"

In [31]:
'''df3 = df3.merge(
    bucket_medians_clean[['walk_dist_bucket', 'PATRON_CATG_DESC_TXT', 'median_time_gap_mins_clean']],
    on=['walk_dist_bucket', 'PATRON_CATG_DESC_TXT'],
    how='left'
)

print(df3['median_time_gap_mins_clean'].isna().sum(), "rows with no clean median")'''

'df3 = df3.merge(\n    bucket_medians_clean[[\'walk_dist_bucket\', \'PATRON_CATG_DESC_TXT\', \'median_time_gap_mins_clean\']],\n    on=[\'walk_dist_bucket\', \'PATRON_CATG_DESC_TXT\'],\n    how=\'left\'\n)\n\nprint(df3[\'median_time_gap_mins_clean\'].isna().sum(), "rows with no clean median")'

In [32]:
#dd useful temporal columns from df3 into df4
temporal_cols = [
    'CRD_NUM',
    'ENTRY_TM',
    'EXIT_TM',
    'time_gap_mins',
    'predicted_walking_time_mins',
    'waiting_time_plus_extra',
    'exceeds_time_allowance',
    'temporal_flag_reason'
]

df4 = df4.merge(
    df3[temporal_cols],
    on=['CRD_NUM', 'ENTRY_TM', 'EXIT_TM'],
    how='left'
)

#combined rule terminate if binary OR temporal says so
df4['final_termination_flag'] = (
    df4['journey_termination_flag'] |
    df4['exceeds_time_allowance']
)

df4['final_termination_reason'] = ''

#binary
df4.loc[df4['journey_termination_flag'], 'final_termination_reason'] += (
    df4['journey_termination_reason'].fillna('') + ' | '
)

#temporal
df4.loc[df4['exceeds_time_allowance'], 'final_termination_reason'] += (
    df4['temporal_flag_reason'].fillna('') + ' | '
)

#clean trailing separator
df4['final_termination_reason'] = (
    df4['final_termination_reason']
    .str.rstrip(' | ')
)

#default
df4.loc[df4['final_termination_reason'] == '', 'final_termination_reason'] = 'candidate_transfer'

print(df4['final_termination_flag'].value_counts(dropna=False))
print(df4['final_termination_reason'].value_counts(dropna=False))

final_termination_flag
True     5796000
False    1723111
Name: count, dtype: int64
final_termination_reason
last_stage | null_time_gap                              2464637
candidate_transfer                                      1723111
return_or_intermediate | exceeds_total_allowance        1456864
exceeds_total_allowance                                 1035758
return_or_intermediate                                   448859
null_predicted_walking_time                              231337
return_or_intermediate | null_predicted_walking_time      79367
missing_info | null_time_gap                              79033
missing_info | null_predicted_walking_time                   59
missing_info                                                 53
missing_info | exceeds_total_allowance                       33
Name: count, dtype: int64


In [33]:
'''df3['temporal_flag_reason_clean'] = np.select(
    [
        df3['time_gap_mins'].isna(),
        df3['median_time_gap_mins_clean'].isna(),
        df3['time_gap_mins'] > df3['median_time_gap_mins_clean'],
    ],
    [
        'null_time_gap',
        'null_median_no_bucket_or_patron',
        'exceeds_bucket_median',
    ],
    default='within_median'
)

df3['exceeds_time_allowance_clean'] = df3['temporal_flag_reason_clean'] != 'within_median'

print(df3['temporal_flag_reason_clean'].value_counts(dropna=False))'''

"df3['temporal_flag_reason_clean'] = np.select(\n    [\n        df3['time_gap_mins'].isna(),\n        df3['median_time_gap_mins_clean'].isna(),\n        df3['time_gap_mins'] > df3['median_time_gap_mins_clean'],\n    ],\n    [\n        'null_time_gap',\n        'null_median_no_bucket_or_patron',\n        'exceeds_bucket_median',\n    ],\n    default='within_median'\n)\n\ndf3['exceeds_time_allowance_clean'] = df3['temporal_flag_reason_clean'] != 'within_median'\n\nprint(df3['temporal_flag_reason_clean'].value_counts(dropna=False))"

#### 3. Spatial Criteria (Reasonable walking dist vs Actual walking dist)
- The spatial rule identifies whether the transition between consecutive journey stages is likely a transfer or the start of a new journey based on walking distance.
- It flags if the distance between the alighting location of the first journey stage and the boarding location of the next journey stage exceeds a reasonable transfer walking distance.
- Implementation: 
    - Reasonable transfer wwalking distance is the column, walking distance
    - Actual walking distance is time gap between tap out and the next tap in * walking speed
    - To account for time where commuter might be waiting for the next ride instead of walking, we include a 20% buffer.

In [34]:
'''# actual walking distance (m) = time gap (seconds) × walking speed (m/s)
# time_gap_mins already exists from temporal step, convert back to seconds
df3['actual_walking_dist_m'] = (
    df3['time_gap_mins'] * 60
) * df3['walking_speed_ms']

# Allowance = walk_distance (m) with 20% buffer
# If walk_distance is null → fillna(True) flags as new journey
df3['exceeds_walking_dist'] = (
    df3['actual_walking_dist_m'] > df3['walk_distance'] * 1.2
).fillna(True)

print(df3['exceeds_walking_dist'].value_counts())
print(f"\nTotal spatial new journey flags: {df3['exceeds_walking_dist'].sum():,}")'''

'# actual walking distance (m) = time gap (seconds) × walking speed (m/s)\n# time_gap_mins already exists from temporal step, convert back to seconds\ndf3[\'actual_walking_dist_m\'] = (\n    df3[\'time_gap_mins\'] * 60\n) * df3[\'walking_speed_ms\']\n\n# Allowance = walk_distance (m) with 20% buffer\n# If walk_distance is null → fillna(True) flags as new journey\ndf3[\'exceeds_walking_dist\'] = (\n    df3[\'actual_walking_dist_m\'] > df3[\'walk_distance\'] * 1.2\n).fillna(True)\n\nprint(df3[\'exceeds_walking_dist\'].value_counts())\nprint(f"\nTotal spatial new journey flags: {df3[\'exceeds_walking_dist\'].sum():,}")'

## Hard geographic threshold (consider)

In [35]:
SPATIAL_THRESHOLD_M = 1200

df3['exceeds_walking_dist_threshold'] = (
    df3['walk_distance'] > SPATIAL_THRESHOLD_M
)

df3['spatial_flag_reason'] = np.select(
    [
        df3['walk_distance'].isna(),
        df3['walk_distance'] > SPATIAL_THRESHOLD_M
    ],
    [
        'null_walk_distance',
        'exceeds_distance_threshold'
    ],
    default='within_distance_threshold'
)

print(df3['exceeds_walking_dist_threshold'].value_counts(dropna=False))
print(df3['spatial_flag_reason'].value_counts(dropna=False))

exceeds_walking_dist_threshold
False    7377923
True      105308
Name: count, dtype: int64
spatial_flag_reason
within_distance_threshold     4713999
null_walk_distance            2663924
exceeds_distance_threshold     105308
Name: count, dtype: int64


In [36]:
'''df3['spatial_flag_reason'] = np.select(
    [
        df3['walk_distance'].isna(),
        df3['walk_distance'] > SPATIAL_THRESHOLD_M,
    ],
    [
        'null_walk_distance',
        'exceeds_walking_dist_threshold',
    ],
    default='within_distance_threshold'
)

df3['exceeds_walking_dist_threshold'] = df3['spatial_flag_reason'] != 'within_distance_threshold'

print(df3['spatial_flag_reason'].value_counts(dropna=False))'''

"df3['spatial_flag_reason'] = np.select(\n    [\n        df3['walk_distance'].isna(),\n        df3['walk_distance'] > SPATIAL_THRESHOLD_M,\n    ],\n    [\n        'null_walk_distance',\n        'exceeds_walking_dist_threshold',\n    ],\n    default='within_distance_threshold'\n)\n\ndf3['exceeds_walking_dist_threshold'] = df3['spatial_flag_reason'] != 'within_distance_threshold'\n\nprint(df3['spatial_flag_reason'].value_counts(dropna=False))"

In [37]:
df3['spatial_full_journey_seq'] = (
    df3.groupby('CRD_NUM')['exceeds_walking_dist_threshold']
       .shift(fill_value=False)
       .groupby(df3['CRD_NUM'])
       .cumsum()
)

#first row of each spatial-only full journey
spatial_journey_first = (
    df3.drop_duplicates(subset=['CRD_NUM', 'spatial_full_journey_seq'], keep='first')
       [['CRD_NUM', 'spatial_full_journey_seq', 'ORIG_STATION_NAME']]
       .rename(columns={
           'ORIG_STATION_NAME': '(SPATIAL_Only)_orig_journey'
       })
)

#last row of each spatial-only full journey
spatial_journey_last = (
    df3.drop_duplicates(subset=['CRD_NUM', 'spatial_full_journey_seq'], keep='last')
       [['CRD_NUM', 'spatial_full_journey_seq', 'DEST_STATION_NAME']]
       .rename(columns={
           'DEST_STATION_NAME': '(SPATIAL_Only)_dest_journey'
       })
)

#combine first + last journey info
spatial_journey_info = spatial_journey_first.merge(
    spatial_journey_last,
    on=['CRD_NUM', 'spatial_full_journey_seq'],
    how='inner'
)

#merge back to every row in df3
df3 = df3.merge(
    spatial_journey_info,
    on=['CRD_NUM', 'spatial_full_journey_seq'],
    how='left'
)

In [38]:
df3['(SPATIAL_Only)_short_round_trip_flag'] = (
    (df3['(SPATIAL_Only)_orig_journey'] == df3['(SPATIAL_Only)_dest_journey']) &
    df3['(SPATIAL_Only)_orig_journey'].notna() &
    df3['(SPATIAL_Only)_dest_journey'].notna()
)

df3['(SPATIAL_Only)_round_trip_reason'] = np.select(
    [
        df3['(SPATIAL_Only)_orig_journey'].isna() |
        df3['(SPATIAL_Only)_dest_journey'].isna(),

        df3['(SPATIAL_Only)_orig_journey'] == df3['(SPATIAL_Only)_dest_journey']
    ],
    [
        'null_origin_or_destination',
        'same_origin_and_destination'
    ],
    default='different_origin_destination'
)

print(df3['(SPATIAL_Only)_short_round_trip_flag'].value_counts(dropna=False))
print(df3['(SPATIAL_Only)_round_trip_reason'].value_counts(dropna=False))

(SPATIAL_Only)_short_round_trip_flag
False    5648891
True     1834340
Name: count, dtype: int64
(SPATIAL_Only)_round_trip_reason
different_origin_destination    5616988
same_origin_and_destination     1834340
null_origin_or_destination        31903
Name: count, dtype: int64


In [39]:
df3['spatial_break_flag_final'] = (
    df3['exceeds_walking_dist_threshold'] |
    df3['(SPATIAL_Only)_short_round_trip_flag']
)

print(df3['spatial_break_flag_final'].value_counts(dropna=False))

spatial_break_flag_final
False    5545738
True     1937493
Name: count, dtype: int64


### Combining all 3 criteria

In [40]:
spatial_cols = [
    'CRD_NUM',
    'ENTRY_TM',
    'EXIT_TM',
    'exceeds_walking_dist_threshold',
    'spatial_break_flag_final'
]

df4 = df4.merge(
    df3[spatial_cols],
    on=['CRD_NUM', 'ENTRY_TM', 'EXIT_TM'],
    how='left'
)

#bin + temp
df4['final_termination_flag'] = (
    df4['journey_termination_flag'] |
    df4['exceeds_time_allowance']
)

df4['final_termination_reason'] = ''

#binary
df4.loc[df4['journey_termination_flag'], 'final_termination_reason'] += (
    df4['journey_termination_reason'].fillna('') + ' | '
)

#temporal
df4.loc[df4['exceeds_time_allowance'], 'final_termination_reason'] += (
    df4['temporal_flag_reason'].fillna('') + ' | '
)

#clean trailing separator
df4['final_termination_reason'] = (
    df4['final_termination_reason']
    .str.rstrip(' | ')
)

#default
df4.loc[df4['final_termination_reason'] == '', 'final_termination_reason'] = 'candidate_transfer'

print(df4['final_termination_flag'].value_counts(dropna=False))
print(df4['final_termination_reason'].value_counts(dropna=False))

final_termination_flag
True     5975400
False    1723111
Name: count, dtype: int64
final_termination_reason
last_stage | null_time_gap                              2477177
candidate_transfer                                      1723111
return_or_intermediate | exceeds_total_allowance        1456864
exceeds_total_allowance                                 1035758
return_or_intermediate                                   448859
missing_info | null_time_gap                             245893
null_predicted_walking_time                              231337
return_or_intermediate | null_predicted_walking_time      79367
missing_info | null_predicted_walking_time                   59
missing_info                                                 53
missing_info | exceeds_total_allowance                       33
Name: count, dtype: int64


In [41]:
#build journey sequence using binary + temporal first
df4['final_full_journey_seq'] = (
    df4.groupby('CRD_NUM')['final_termination_flag']
       .shift(fill_value=False)
       .groupby(df4['CRD_NUM'])
       .cumsum()
)

#first row of each binary+temporal journey
final_journey_first = (
    df4.drop_duplicates(subset=['CRD_NUM', 'final_full_journey_seq'], keep='first')
       [['CRD_NUM', 'final_full_journey_seq', 'ORIG_STATION_NAME']]
       .rename(columns={
           'ORIG_STATION_NAME': 'final_orig_journey_station'
       })
)

#last row of each binary+temporal journey
final_journey_last = (
    df4.drop_duplicates(subset=['CRD_NUM', 'final_full_journey_seq'], keep='last')
       [['CRD_NUM', 'final_full_journey_seq', 'DEST_STATION_NAME']]
       .rename(columns={
           'DEST_STATION_NAME': 'final_dest_journey_station'
       })
)

#combine first + last journey info
final_journey_info = final_journey_first.merge(
    final_journey_last,
    on=['CRD_NUM', 'final_full_journey_seq'],
    how='inner'
)

#merge back to df4
df4 = df4.merge(
    final_journey_info,
    on=['CRD_NUM', 'final_full_journey_seq'],
    how='left'
)

In [42]:
#now round trip is based on binary + temporal journeys, not binary-only journeys
df4['short_round_trip_flag'] = (
    (df4['final_orig_journey_station'] == df4['final_dest_journey_station']) &
    df4['final_orig_journey_station'].notna() &
    df4['final_dest_journey_station'].notna()
)

df4['round_trip_reason'] = np.select(
    [
        df4['final_orig_journey_station'].isna() |
        df4['final_dest_journey_station'].isna(),

        df4['final_orig_journey_station'] == df4['final_dest_journey_station']
    ],
    [
        'null_origin_or_destination',
        'same_origin_and_destination'
    ],
    default='different_origin_destination'
)

print(df4['short_round_trip_flag'].value_counts(dropna=False))
print(df4['round_trip_reason'].value_counts(dropna=False))

short_round_trip_flag
False    7681079
True       17432
Name: count, dtype: int64
round_trip_reason
different_origin_destination    7646440
null_origin_or_destination        34639
same_origin_and_destination       17432
Name: count, dtype: int64


In [43]:
df4['final_termination_flag_spatial'] = (
    df4['journey_termination_flag'] |
    df4['exceeds_time_allowance'] |
    df4['exceeds_walking_dist_threshold'] |
    df4['short_round_trip_flag']
)

df4['final_termination_reason_spatial'] = ''

# binary
df4.loc[df4['journey_termination_flag'], 'final_termination_reason_spatial'] += (
    df4['journey_termination_reason'].fillna('') + ' | '
)

# temporal
df4.loc[df4['exceeds_time_allowance'], 'final_termination_reason_spatial'] += (
    df4['temporal_flag_reason'].fillna('') + ' | '
)

# spatial 1
df4.loc[df4['exceeds_walking_dist_threshold'], 'final_termination_reason_spatial'] += (
    'distance_too_far | '
)

# spatial 2
df4.loc[df4['short_round_trip_flag'], 'final_termination_reason_spatial'] += (
    'round_trip_detected | '
)

# clean trailing separator
df4['final_termination_reason_spatial'] = (
    df4['final_termination_reason_spatial']
    .str.rstrip(' | ')
)

# default
df4.loc[df4['final_termination_reason_spatial'] == '', 'final_termination_reason_spatial'] = 'candidate_transfer'

print(df4['final_termination_flag_spatial'].value_counts(dropna=False))
print(df4['final_termination_reason_spatial'].value_counts(dropna=False))

final_termination_flag_spatial
True     5984871
False    1713640
Name: count, dtype: int64
final_termination_reason_spatial
last_stage | null_time_gap                                                    2472689
candidate_transfer                                                            1713640
return_or_intermediate | exceeds_total_allowance                              1450505
exceeds_total_allowance                                                        943213
return_or_intermediate                                                         444440
missing_info | null_time_gap                                                   243625
null_predicted_walking_time                                                    228828
exceeds_total_allowance | distance_too_far                                      91462
return_or_intermediate | null_predicted_walking_time                            79061
distance_too_far                                                                 5650
last_stage | nul

In [44]:
df4['final_termination_flag_spatial_nodist'] = (
    df4['journey_termination_flag'] |
    df4['exceeds_time_allowance'] |
    df4['short_round_trip_flag']
)

df4['final_termination_reason_spatial_nodist'] = ''

# binary
df4.loc[df4['journey_termination_flag'], 'final_termination_reason_spatial_nodist'] += (
    df4['journey_termination_reason'].fillna('') + ' | '
)

# temporal
df4.loc[df4['exceeds_time_allowance'], 'final_termination_reason_spatial_nodist'] += (
    df4['temporal_flag_reason'].fillna('') + ' | '
)

# spatial (round trip only)
df4.loc[df4['short_round_trip_flag'], 'final_termination_reason_spatial_nodist'] += (
    'round_trip_detected | '
)

# clean trailing separator
df4['final_termination_reason_spatial_nodist'] = (
    df4['final_termination_reason_spatial_nodist']
    .str.rstrip(' | ')
)

# default
df4.loc[
    df4['final_termination_reason_spatial_nodist'] == '',
    'final_termination_reason_spatial_nodist'
] = 'candidate_transfer'

print(df4['final_termination_flag_spatial_nodist'].value_counts(dropna=False))
print(df4['final_termination_reason_spatial_nodist'].value_counts(dropna=False))

final_termination_flag_spatial_nodist
True     5979221
False    1719290
Name: count, dtype: int64
final_termination_reason_spatial_nodist
last_stage | null_time_gap                                                    2472689
candidate_transfer                                                            1719290
return_or_intermediate | exceeds_total_allowance                              1453937
exceeds_total_allowance                                                       1034675
return_or_intermediate                                                         444564
missing_info | null_time_gap                                                   245737
null_predicted_walking_time                                                    230892
return_or_intermediate | null_predicted_walking_time                            79150
last_stage | null_time_gap | round_trip_detected                                 4488
return_or_intermediate | round_trip_detected                                     4295
ro

In [45]:
df4['short_round_trip_flag'] = (
    (df4['final_orig_journey_station'] == df4['final_dest_journey_station']) &
    df4['final_orig_journey_station'].notna() &
    df4['final_dest_journey_station'].notna()
)

df4['round_trip_reason'] = np.select(
    [
        df4['final_orig_journey_station'].isna() |
        df4['final_dest_journey_station'].isna(),

        df4['final_orig_journey_station'] == df4['final_dest_journey_station']
    ],
    [
        'null_origin_or_destination',
        'same_origin_and_destination'
    ],
    default='different_origin_destination'
)

print(df4['short_round_trip_flag'].value_counts(dropna=False))
print(df4['round_trip_reason'].value_counts(dropna=False))

short_round_trip_flag
False    7681079
True       17432
Name: count, dtype: int64
round_trip_reason
different_origin_destination    7646440
null_origin_or_destination        34639
same_origin_and_destination       17432
Name: count, dtype: int64


In [46]:
'''# Old spatial
df3['combined_flag1'] = (
    df3['journey_termination_flag'] |
    df3['exceeds_time_allowance'] |
    df3['exceeds_walking_dist']
)

df3['combined_flag1_reason'] = np.select(
    [
        df3['journey_termination_flag'],
        df3['exceeds_time_allowance'],
        df3['exceeds_walking_dist'],
    ],
    [
        'binary',
        'temporal',
        'spatial',
    ],
    default='transfer'
)

print(df3['combined_flag1'].value_counts())
print()
print(df3['combined_flag1_reason'].value_counts(dropna=False))'''

"# Old spatial\ndf3['combined_flag1'] = (\n    df3['journey_termination_flag'] |\n    df3['exceeds_time_allowance'] |\n    df3['exceeds_walking_dist']\n)\n\ndf3['combined_flag1_reason'] = np.select(\n    [\n        df3['journey_termination_flag'],\n        df3['exceeds_time_allowance'],\n        df3['exceeds_walking_dist'],\n    ],\n    [\n        'binary',\n        'temporal',\n        'spatial',\n    ],\n    default='transfer'\n)\n\nprint(df3['combined_flag1'].value_counts())\nprint()\nprint(df3['combined_flag1_reason'].value_counts(dropna=False))"

In [47]:
'''# New spatial
df3['combined_flag2'] = (
    df3['journey_termination_flag'] |
    df3['exceeds_time_allowance'] |
    df3['exceeds_walking_dist_threshold']
)

df3['combined_flag2_reason'] = np.select(
    [
        df3['journey_termination_flag'],
        df3['exceeds_time_allowance'],
        df3['exceeds_walking_dist_threshold'],
    ],
    [
        'binary',
        'temporal',
        'spatial',
    ],
    default='transfer'
)

print(df3['combined_flag2'].value_counts())
print()
print(df3['combined_flag2_reason'].value_counts(dropna=False))'''

"# New spatial\ndf3['combined_flag2'] = (\n    df3['journey_termination_flag'] |\n    df3['exceeds_time_allowance'] |\n    df3['exceeds_walking_dist_threshold']\n)\n\ndf3['combined_flag2_reason'] = np.select(\n    [\n        df3['journey_termination_flag'],\n        df3['exceeds_time_allowance'],\n        df3['exceeds_walking_dist_threshold'],\n    ],\n    [\n        'binary',\n        'temporal',\n        'spatial',\n    ],\n    default='transfer'\n)\n\nprint(df3['combined_flag2'].value_counts())\nprint()\nprint(df3['combined_flag2_reason'].value_counts(dropna=False))"

In [48]:
'''# Binary + Temporal only
df3['combined_flag3'] = (
    df3['journey_termination_flag'] |
    df3['exceeds_time_allowance']
)

df3['combined_flag3_reason'] = np.select(
    [
        df3['journey_termination_flag'],
        df3['exceeds_time_allowance'],
    ],
    [
        'binary',
        'temporal',
    ],
    default='transfer'
)

print(df3['combined_flag3'].value_counts())
print()
print(df3['combined_flag3_reason'].value_counts(dropna=False))'''

"# Binary + Temporal only\ndf3['combined_flag3'] = (\n    df3['journey_termination_flag'] |\n    df3['exceeds_time_allowance']\n)\n\ndf3['combined_flag3_reason'] = np.select(\n    [\n        df3['journey_termination_flag'],\n        df3['exceeds_time_allowance'],\n    ],\n    [\n        'binary',\n        'temporal',\n    ],\n    default='transfer'\n)\n\nprint(df3['combined_flag3'].value_counts())\nprint()\nprint(df3['combined_flag3_reason'].value_counts(dropna=False))"

In [49]:
'''# Binary + clean temporal only
df3['combined_flag4'] = (
    df3['journey_termination_flag'] |
    df3['exceeds_time_allowance_clean']
)

df3['combined_flag4_reason'] = np.select(
    [
        df3['journey_termination_flag'],
        df3['exceeds_time_allowance_clean'],
    ],
    [
        'binary',
        'temporal',
    ],
    default='transfer'
)

print(df3['combined_flag4'].value_counts())
print()
print(df3['combined_flag4_reason'].value_counts(dropna=False))'''

"# Binary + clean temporal only\ndf3['combined_flag4'] = (\n    df3['journey_termination_flag'] |\n    df3['exceeds_time_allowance_clean']\n)\n\ndf3['combined_flag4_reason'] = np.select(\n    [\n        df3['journey_termination_flag'],\n        df3['exceeds_time_allowance_clean'],\n    ],\n    [\n        'binary',\n        'temporal',\n    ],\n    default='transfer'\n)\n\nprint(df3['combined_flag4'].value_counts())\nprint()\nprint(df3['combined_flag4_reason'].value_counts(dropna=False))"

## Internal Validity Check with pt_jrny

In [50]:
df5 = pd.read_pickle('../data/df5.pkl')
df5.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5336605 entries, 0 to 5336604
Data columns (total 23 columns):
 #   Column              Dtype         
---  ------              -----         
 0   CRD_NUM             object        
 1   JRNY_DEST_ID_NUM    float64       
 2   JRNY_START_DT       datetime64[ns]
 3   JRNY_START_TM       datetime64[ns]
 4   JRNY_END_DT         datetime64[ns]
 5   JRNY_END_TM         datetime64[ns]
 6   JRNY_ORIG_ID_NUM    float64       
 7   JRNY_DIST_KM_CNT    float64       
 8   JRNY_FARE_AMT       float64       
 9   JRNY_ID_NUM         object        
 10  JRNY_TM_MIN_CNT     float64       
 11  PATRON_CATG_ID_NUM  float64       
 12  TRNSPT_MODE_CD      int64         
 13  DEST_STATION_NAME   object        
 14  DEST_MRK_ID_NUM     float64       
 15  DEST_LATITUDE       float64       
 16  DEST_LONGITUDE      float64       
 17  DEST_Travel_Type    object        
 18  ORIG_STATION_NAME   object        
 19  ORIG_MRK_ID_NUM     float64       
 20  OR

In [51]:
# same filtering as df3
df5 = df5[df5['PATRON_CATG_ID_NUM'].isin([1, 3, 4])].reset_index(drop=True)

In [ ]:
# Merge on CRD_NUM + JRNY_ID_NUM to attach journey boundaries to each ride
df_val = df4.merge(
    df5[['CRD_NUM', 'JRNY_ID_NUM', 'JRNY_START_TM', 'JRNY_END_TM']],
    on=['CRD_NUM', 'JRNY_ID_NUM'],
    how='inner'
)

# Keep only rides that fall within their journey's time window
df_val = df_val[
    (df_val['ENTRY_TM'] >= df_val['JRNY_START_TM']) &
    (df_val['EXIT_TM'] <= df_val['JRNY_END_TM'])
].reset_index(drop=True)

print(f"Rows after merge + time filter: {len(df_val):,}")

In [ ]:
df_val = df_val.sort_values(['CRD_NUM', 'ENTRY_TM']).reset_index(drop=True)
# Shift JRNY_ID_NUM within each commuter to get the next ride's journey ID
df_val['next_JRNY_ID_NUM'] = df_val.groupby('CRD_NUM')['JRNY_ID_NUM'].shift(-1)

# true_transfer = True → same journey (actual transfer)
# true_transfer = False → different journey (actual new journey)
df_val['true_transfer'] = (df_val['JRNY_ID_NUM'] == df_val['next_JRNY_ID_NUM'])

# Drop last stage of each commuter — no next ride to compare against
df_val = df_val[df_val['next_JRNY_ID_NUM'].notna()].reset_index(drop=True)

print(df_val['true_transfer'].value_counts())

true_transfer
False    2610552
True     2226129
Name: count, dtype: int64


In [ ]:
# helper function
def print_metrics(df, pred_flag_col, label):
    """
    pred_flag_col: True = classifier says NEW JOURNEY, False = classifier says TRANSFER
    true_transfer: True = actual transfer, False = actual new journey
    """
    actual_transfer = df['true_transfer']
    pred_transfer = ~df[pred_flag_col]      # flip: flag=True means new journey, so ~flag = predicted transfer

    tp = (actual_transfer & pred_transfer).sum()     # actual transfer, predicted transfer
    tn = (~actual_transfer & ~pred_transfer).sum()   # actual new journey, predicted new journey
    fp = (~actual_transfer & pred_transfer).sum()    # actual new journey, predicted transfer (merge error)
    fn = (actual_transfer & ~pred_transfer).sum()    # actual transfer, predicted new journey (split error)

    total_actual_transfers = actual_transfer.sum()

    print(f"\n=== {label} ===")
    print(f"TP: {tp:,}  TN: {tn:,}  FP: {fp:,}  FN: {fn:,}")
    print(f"Split rate  (FN / actual transfers): {fn / total_actual_transfers:.4f}")
    print(f"Merge rate  (FP / actual transfers): {fp / total_actual_transfers:.4f}")
    print(f"Sensitivity (TP / (TP+FN)):          {tp / (tp + fn):.4f}")
    print(f"Specificity (TN / (TN+FP)):          {tn / (tn + fp):.4f}")
    print(f"Accuracy    ((TP+TN) / total):       {(tp + tn) / len(df):.4f}")

    print(pd.crosstab(
        actual_transfer,
        pred_transfer,
        rownames=['Actual transfer'],
        colnames=[f'Predicted transfer ({label})']
    ))

In [ ]:
print_metrics(df_val, 'journey_termination_flag', 'Binary Criteria')


=== Binary Criteria ===
TP: 1,851,492  TN: 1,551,221  FP: 1,059,331  FN: 374,637
Split rate  (FN / actual transfers): 0.1683
Merge rate  (FP / actual transfers): 0.4759
Sensitivity (TP / (TP+FN)):          0.8317
Specificity (TN / (TN+FP)):          0.5942
Accuracy    ((TP+TN) / total):       0.7035
Predicted transfer (Binary Criteria)    False    True 
Actual transfer                                       
False                                 1551221  1059331
True                                   374637  1851492


In [ ]:
print_metrics(df_val, 'exceeds_time_allowance', 'Temporal Criteria')


=== Temporal Criteria ===
TP: 2,042,953  TN: 2,445,062  FP: 165,490  FN: 183,176
Split rate  (FN / actual transfers): 0.0823
Merge rate  (FP / actual transfers): 0.0743
Sensitivity (TP / (TP+FN)):          0.9177
Specificity (TN / (TN+FP)):          0.9366
Accuracy    ((TP+TN) / total):       0.9279
Predicted transfer (Temporal Criteria)    False    True 
Actual transfer                                         
False                                   2445062   165490
True                                     183176  2042953


In [ ]:
print_metrics(df_val, 'exceeds_walking_dist_threshold', 'Spatial Criteria (Distance Threshold)')


=== Spatial Criteria (Distance Threshold) ===
TP: 2,216,660  TN: 193,401  FP: 2,417,151  FN: 9,469
Split rate  (FN / actual transfers): 0.0043
Merge rate  (FP / actual transfers): 1.0858
Sensitivity (TP / (TP+FN)):          0.9957
Specificity (TN / (TN+FP)):          0.0741
Accuracy    ((TP+TN) / total):       0.4983
Predicted transfer (Spatial Criteria (Distance Threshold))   False    True 
Actual transfer                                                            
False                                                       193401  2417151
True                                                          9469  2216660


In [ ]:
print_metrics(df_val, 'short_round_trip_flag', 'Spatial Criteria (Round Trip)')


=== Spatial Criteria (Round Trip) ===
TP: 2,222,554  TN: 9,513  FP: 2,601,039  FN: 3,575
Split rate  (FN / actual transfers): 0.0016
Merge rate  (FP / actual transfers): 1.1684
Sensitivity (TP / (TP+FN)):          0.9984
Specificity (TN / (TN+FP)):          0.0036
Accuracy    ((TP+TN) / total):       0.4615
Predicted transfer (Spatial Criteria (Round Trip))  False    True 
Actual transfer                                                   
False                                                9513  2601039
True                                                 3575  2222554


In [ ]:
print_metrics(df_val, 'spatial_break_flag_final', 'Spatial Criteria (Combined)')


=== Spatial Criteria (Combined) ===
TP: 1,817,954  TN: 952,294  FP: 1,658,258  FN: 408,175
Split rate  (FN / actual transfers): 0.1834
Merge rate  (FP / actual transfers): 0.7449
Sensitivity (TP / (TP+FN)):          0.8166
Specificity (TN / (TN+FP)):          0.3648
Accuracy    ((TP+TN) / total):       0.5728
Predicted transfer (Spatial Criteria (Combined))   False    True 
Actual transfer                                                  
False                                             952294  1658258
True                                              408175  1817954


In [ ]:
print_metrics(df_val, 'final_termination_flag', 'Binary + Temporal')


=== Binary + Temporal ===
TP: 1,693,457  TN: 2,548,896  FP: 61,656  FN: 532,672
Split rate  (FN / actual transfers): 0.2393
Merge rate  (FP / actual transfers): 0.0277
Sensitivity (TP / (TP+FN)):          0.7607
Specificity (TN / (TN+FP)):          0.9764
Accuracy    ((TP+TN) / total):       0.8771
Predicted transfer (Binary + Temporal)    False    True 
Actual transfer                                         
False                                   2548896    61656
True                                     532672  1693457


In [ ]:
print_metrics(df_val, 'final_termination_flag_spatial', 'Binary + Temporal + Spatial (With Distance Threshold)')


=== Binary + Temporal + Spatial (With Distance Threshold) ===
TP: 1,681,429  TN: 2,558,344  FP: 52,208  FN: 544,700
Split rate  (FN / actual transfers): 0.2447
Merge rate  (FP / actual transfers): 0.0235
Sensitivity (TP / (TP+FN)):          0.7553
Specificity (TN / (TN+FP)):          0.9800
Accuracy    ((TP+TN) / total):       0.8766
Predicted transfer (Binary + Temporal + Spatial (With Distance Threshold))    False  \
Actual transfer                                                                       
False                                                                       2558344   
True                                                                         544700   

Predicted transfer (Binary + Temporal + Spatial (With Distance Threshold))    True   
Actual transfer                                                                      
False                                                                         52208  
True                                                    

In [ ]:
print_metrics(df_val, 'final_termination_flag_spatial_nodist', 'Binary + Temporal + Spatial (Round Trip Only)')


=== Binary + Temporal + Spatial (Round Trip Only) ===
TP: 1,690,127  TN: 2,550,153  FP: 60,399  FN: 536,002
Split rate  (FN / actual transfers): 0.2408
Merge rate  (FP / actual transfers): 0.0271
Sensitivity (TP / (TP+FN)):          0.7592
Specificity (TN / (TN+FP)):          0.9769
Accuracy    ((TP+TN) / total):       0.8767
Predicted transfer (Binary + Temporal + Spatial (Round Trip Only))    False  \
Actual transfer                                                               
False                                                               2550153   
True                                                                 536002   

Predicted transfer (Binary + Temporal + Spatial (Round Trip Only))    True   
Actual transfer                                                              
False                                                                 60399  
True                                                                1690127  


# journey dataset that imposes all criterias

In [ ]:
df4 = df4.sort_values(['CRD_NUM', 'ENTRY_TM']).reset_index(drop=True)

# define final journey sequence:
# a row with final_termination_flag_spatial = True ends the current journey
df4['final_journey_seq'] = (
    df4.groupby('CRD_NUM')['final_termination_flag_spatial']
       .shift(fill_value=False)
       .groupby(df4['CRD_NUM'])
       .cumsum()
)

# first row of each final journey
journey_first = (
    df4.drop_duplicates(subset=['CRD_NUM', 'final_journey_seq'], keep='first')
       [['CRD_NUM', 'final_journey_seq', 'ORIG_STATION_NAME', 'ENTRY_TM']]
       .rename(columns={
           'ORIG_STATION_NAME': 'journey_orig_station',
           'ENTRY_TM': 'journey_entry_tm'
       })
)

# last row of each final journey
journey_last = (
    df4.drop_duplicates(subset=['CRD_NUM', 'final_journey_seq'], keep='last')
       [['CRD_NUM', 'final_journey_seq', 'DEST_STATION_NAME', 'EXIT_TM',
         'walk_distance', 'next_orig_station']]
       .rename(columns={
           'DEST_STATION_NAME': 'journey_dest_station',
           'EXIT_TM': 'journey_exit_tm',
           'walk_distance': 'walk_to_next_journey_distance',
           'next_orig_station': 'next_journey_orig_station'
       })
)

# one row per final journey
df_journey = journey_first.merge(
    journey_last,
    on=['CRD_NUM', 'final_journey_seq'],
    how='inner'
).drop_duplicates(subset=['CRD_NUM', 'final_journey_seq']).reset_index(drop=True)

#print(df_journey.head())
print(df_journey.shape)
print(df_journey[['CRD_NUM', 'final_journey_seq']].duplicated().sum(), "duplicate journey keys")

(5977127, 8)
0 duplicate journey keys


In [ ]:
# save df4 as a pickle for welfare analysis and delay simulation
df4.to_pickle('../data/df4.pkl')
print('Saved df4.pkl')

SyntaxError: invalid syntax (1085795263.py, line 1)

In [ ]:
'''# Temporal after filtering out binary
print_metrics(df_val, 'exceeds_time_allowance_clean', 'Temporal Criteria (Clean Median)')'''

In [ ]:
'''# first spatial
print_metrics(df_val, 'exceeds_walking_dist', 'Spatial Criteria')'''

In [ ]:
'''# second spatial
print_metrics(df_val, 'exceeds_walking_dist_threshold', 'Spatial Criteria')'''

- combined_flag1: all 3 original criteria
- combined_flag2: all 3, with spatial using a hard threshold
- combined_flag3: binary and temporal flags only
- combined_flag4: binary + temporal after taking into account bin

In [ ]:
'''print_metrics(df_val, 'combined_flag1', 'Combined Criteria')'''

In [ ]:
'''print_metrics(df_val, 'combined_flag2', 'Combined Criteria')'''

In [ ]:
'''print_metrics(df_val, 'combined_flag3', 'Combined Criteria')'''

In [ ]:
'''print_metrics(df_val, 'combined_flag4', 'Combined Flag 4 (Binary + Clean Temporal)')'''

In [ ]:
'''# How much of df_val has null walk_distance?
print("Null walk_distance in df_val:")
print(df_val['walk_distance'].isna().sum(), '/', len(df_val))
print(f"{df_val['walk_distance'].isna().mean()*100:.1f}%")

# Of the true transfers, how many have null walk_distance?
true_transfers = df_val[df_val['true_transfer'] == True]
print("\nNull walk_distance among actual transfers:")
print(true_transfers['walk_distance'].isna().sum(), '/', len(true_transfers))
print(f"{true_transfers['walk_distance'].isna().mean()*100:.1f}%")'''

# ignore below

#### 3. Spatial Criteria (Haversine):
- The spatial rule identifies whether the transition between consecutive journey stages is likely a transfer or the start of a new journey based on walking distance.
- Implementation: 
    - Use original clean df to access logtitude and latitude columns
    - Sort the dataset by card number (CRD_NUM) and timestamp (ENTRY_TM) to get consecutive stages for each rider.
    - For each ride, we look at the next stage’s boarding location (latitude and longitude). Rows without a next stage are ignored because there is nothing to compare. The person finished all their trips for the day. 
    - We create a boolean mask to select only rows that have a next stage (NEXT_ENTRY_LAT is not NaN). This helps us to ignore observations without next stages
    - Use Haversine formula to compute the shortest straight-line distance between DEST_LATITUDE & DEST_LONGTITUDE (alighting stop) and the NEXT_ENTRY_LONG & NEXT_ENTRY_LAT (the next entry point). Vectorisation is used because row-by-row computation was too slow. 
    - Allow a maximum transfer distance of 500 metres (we can change this)
    - If the distance exceeds a maximum allowed transfer distance (0.5 km), the row is flagged as spatial_new_journey = True. This indicates that the next ride is likely a new journey.

In [ ]:
# Shift boarding coordinates to get "next stage" per rider
df3['NEXT_ENTRY_LAT'] = df3.groupby('CRD_NUM')['ORIG_LATITUDE'].shift(-1)
df3['NEXT_ENTRY_LON'] = df3.groupby('CRD_NUM')['ORIG_LONGITUDE'].shift(-1)

In [ ]:
import numpy as np
#use vectorized function so it runs much faster than row by row
def haversine_vectorized(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    
    a = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    c = 2 * np.arcsin(np.sqrt(a))
    
    r = 6371  # Earth radius in km
    return c * r

In [ ]:
#compute distance to next stage if it exists
mask = df3['NEXT_ENTRY_LAT'].notna()  # skip last stage (ie. if there is no next tap in) 
df3.loc[mask, 'distance_to_next_km'] = haversine_vectorized(
    df3.loc[mask, 'DEST_LATITUDE'].values,
    df3.loc[mask, 'DEST_LONGITUDE'].values,
    df3.loc[mask, 'NEXT_ENTRY_LAT'].values,
    df3.loc[mask, 'NEXT_ENTRY_LON'].values
)

In [ ]:
#flag spatial transfers
# set max reasonable walking transfer distance = 0.5 km 
# we can change this threshold
max_transfer_distance_km = 0.5

#initialise as False. Only mark as True if distance to next stage exceeds threshold
df3['spatial_new_journey'] = False
df3.loc[mask, 'spatial_new_journey'] = df3.loc[mask, 'distance_to_next_km'] > max_transfer_distance_km

In [ ]:
df3['spatial_new_journey'].value_counts()

# Internal Validity Check
- Drop rows with missing critical data
- Combine `pt_ride` and `pt_jrny` and only keep those where the ride entry time >= journey start time and ride exit time <= journey end time
- Generate confusion matrix calculate metrics (split rate, merge rate, sensitivity, specificity, accuracy)

In [ ]:
df5 = pd.read_pickle('../data/df5.pkl')

In [ ]:
df5.info()

In [ ]:
# Drop rows with missing critical data
critical_cols = [
    'JRNY_START_TM', 'JRNY_END_TM',
    'JRNY_ORIG_ID_NUM', 'JRNY_DEST_ID_NUM',
    'ORIG_LATITUDE', 'ORIG_LONGITUDE',
    'DEST_LATITUDE', 'DEST_LONGITUDE'
]

before = len(df5)
df5 = df5.dropna(subset=critical_cols).reset_index(drop=True)
after = len(df5)

print(f"Rows dropped: {before - after:,}")
print(f"Rows remaining: {after:,}")

In [ ]:
df4 = df5[df5['JRNY_START_DT'] == '2025-02-11']

#### 1. Binary Criteria

In [ ]:
# Combine pt_ride and pt_jrny and only keep those where the ride entry time = 
df6 = df2.merge(df4, on=['CRD_NUM', 'JRNY_ID_NUM'], suffixes=('', '_JRNY'))
df6 = df6[(df6['ENTRY_DT'] >= df6['JRNY_START_DT']) & (df6['EXIT_DT'] <= df6['JRNY_END_DT'])]

In [ ]:
# Shift needed columns
df6['next_JRNY_ID_NUM'] = df6['JRNY_ID_NUM'].shift(-1)

# Flag true single journeys
df6['true_same_journey'] = (df6['JRNY_ID_NUM'] == df6['next_JRNY_ID_NUM'])

print(df6['true_same_journey'].value_counts())

In [ ]:
# Confusion matrix
pd.crosstab(
    df6['true_same_journey'],
    df6['same_service_return'],
    rownames=['Actual'],
    colnames=['Predicted']
)

In [ ]:
total_true = (df6['true_same_journey'] == True).sum()

# Split Error Rate (FN)
split_error = ((df6['true_same_journey'] == True) & (df6['same_service_return'] == False)).sum()
split_rate = split_error / total_true
print("Split rate:", split_rate)

# Merge Error Rate (FP)
merge_error = ((df6['true_same_journey'] == False) & (df6['same_service_return'] == True)).sum()
merge_rate = merge_error / total_true
print("Merge rate:", merge_rate)

# Sensitivity (TP / (TP + FN))
tp = ((df6['true_same_journey'] == True) & (df6['same_service_return'] == True)).sum()
sensitivity = tp / (tp + split_error)
print("Sensitivity:", sensitivity)

# Specificity (TN / (TN + FP))
tn = ((df6['true_same_journey'] == False) & (df6['same_service_return'] == False)).sum()
specificity = tn / (tn + merge_error)
print("Specificity", specificity)

# Accuracy
accuracy = (tp + tn) / (total_true + merge_error + tn)
print("Accuracy:", accuracy)

#### 2. Temporal Criteria

In [ ]:
df6['true_new_journey'] = (df6['JRNY_ID_NUM'] != df6['next_JRNY_ID_NUM'])

In [ ]:
# Confusion matrix
pd.crosstab(
    df6['true_new_journey'],
    df6['temporal_new_journey'],
    rownames=['Actual'],
    colnames=['Predicted']
)

In [ ]:
# Split Error Rate (FN)
split_error = ((df6['true_new_journey'] == True) & (df6['temporal_new_journey'] == False)).sum()
split_rate = split_error / total_true
print("Split rate:", split_rate)

# Merge Error Rate (FP)
merge_error = ((df6['true_new_journey'] == False) & (df6['temporal_new_journey'] == True)).sum()
merge_rate = merge_error / total_true
print("Merge rate:", merge_rate)

# Sensitivity (TP / (TP + FN))
tp = ((df6['true_new_journey'] == True) & (df6['temporal_new_journey'] == True)).sum()
sensitivity = tp / (tp + split_error)
print("Sensitivity:", sensitivity)

# Specificity (TN / (TN + FP))
tn = ((df6['true_new_journey'] == False) & (df6['temporal_new_journey'] == False)).sum()
specificity = tn / (tn + merge_error)
print("Specificity", specificity)

# Accuracy
accuracy = (tp + tn) / (total_true + merge_error + tn)
print("Accuracy:", accuracy)

#### 3. Spatial Criteria

In [ ]:
# Combine pt_ride and pt_jrny and only keep those where the ride entry time = 
df7 = df3.merge(df5, on=['CRD_NUM', 'JRNY_ID_NUM'], suffixes=('', '_JRNY'))
df7 = df7[(df7['ENTRY_DT'] >= df7['JRNY_START_DT']) & (df7['EXIT_DT'] <= df7['JRNY_END_DT'])]

In [ ]:
# Shift needed columns
df7['next_JRNY_ID_NUM'] = df7['JRNY_ID_NUM'].shift(-1)

# Flag true single journeys
df7['true_same_journey'] = (df7['JRNY_ID_NUM'] == df7['next_JRNY_ID_NUM'])

print(df7['true_same_journey'].value_counts())

In [ ]:
# Confusion matrix
pd.crosstab(
    df7['true_new_journey'],
    df7['spatial_new_journey'],
    rownames=['Actual'],
    colnames=['Predicted']
)

In [ ]:
total_true2 = (df7['true_same_journey'] == True).sum()

# Split Error Rate (FN)
split_error = ((df7['true_new_journey'] == True) & (df7['spatial_new_journey'] == False)).sum()
split_rate = split_error / total_true2
print("Split rate:", split_rate)

# Merge Error Rate (FP)
merge_error = ((df7['true_new_journey'] == False) & (df7['spatial_new_journey'] == True)).sum()
merge_rate = merge_error / total_true2
print("Merge rate:", merge_rate)

# Sensitivity (TP / (TP + FN))
tp = ((df7['true_new_journey'] == True) & (df7['spatial_new_journey'] == True)).sum()
sensitivity = tp / (tp + split_error)
print("Sensitivity:", sensitivity)

# Specificity (TN / (TN + FP))
tn = ((df7['true_new_journey'] == False) & (df7['spatial_new_journey'] == False)).sum()
specificity = tn / (tn + merge_error)
print("Specificity", specificity)

# Accuracy
accuracy = (tp + tn) / (total_true2 + merge_error + tn)
print("Accuracy:", accuracy)